# ReWild: reasoning-guided GRPO from the saved SFT adapter
20 balanced groups: 10 existing Llama-teacher training examples + 10 fresh RL training examples. No SFT retraining or Ollama calls. A gated reference-overlap reward is a heuristic, not process verification. To compare fairly, rerun from the same SFT checkpoint with `REASONING_WEIGHT=0.0` and a different output folder. Both runs use the same question IDs and decoding settings. Keep held-out test questions out of model selection.

In [1]:
import os
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
import gc, json, math, random, re
from collections import Counter, defaultdict
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

SEED = 42
MODEL_ID = 'Qwen/Qwen3-0.6B-Base'
DATA_SOURCE = Path('rewild_mps_local_v2')
SFT_PATH = Path('rewild_mps_llama_distillation/sft_best')
OUT = Path('rewild_reasoning_guided_grpo_v1')
MAX_SEQ_LEN = 2048
EVAL_BATCH = 2
EVAL_MAX_NEW_TOKENS = 512
GRPO_MAX_NEW_TOKENS = 512
GRPO_STEPS = 20
GRPO_ROLLOUTS = 4
TEACHER_PER_TYPE = 2
FRESH_PER_TYPE = 2
GRPO_LR = 2e-6
GRPO_KL_BETA = 0.05
GRPO_CLIP_EPS = 0.2
GRPO_OPT_EPOCHS = 1
GRPO_VAL_EVERY = 10
FORMAT_WEIGHT = 0.05
REASONING_WEIGHT = 0.15  # 0.0 for matched mixed-data outcome-only control
REUSE_SFT_VALIDATION = True  # only if original checkpoint, prompts and eval settings match
RUN_TEST = False  # set True after freezing all experimental settings
assert GRPO_STEPS == 5 * (TEACHER_PER_TYPE + FRESH_PER_TYPE)
assert GRPO_ROLLOUTS >= 2 and 0 <= REASONING_WEIGHT < 1-FORMAT_WEIGHT
random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float32
assert not OUT.exists() or not any(OUT.iterdir()), 'Choose a new OUT folder to avoid overwriting experiments.'
OUT.mkdir(parents=True, exist_ok=True)
def save_json(name, obj):
    path = OUT / name; tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8'); tmp.replace(path)
    return path
def empty_cache():
    gc.collect()
    if device.type == 'mps': torch.mps.empty_cache()
    elif device.type == 'cuda': torch.cuda.empty_cache()
print('Device:', device, 'Output:', OUT)


/opt/anaconda3/envs/rewild_mps/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps Output: rewild_reasoning_guided_grpo_v1


## Load saved training splits and the original SFT adapter. No regeneration.

In [2]:
def read_split(name):
    return json.loads((DATA_SOURCE / name).read_text(encoding='utf-8'))
train_sft = read_split('sft_llama_teacher.json')
rl_data = read_split('rl_data.json')
val_data = read_split('validation_data.json')
test_data = read_split('test_data.json')
sft_data = train_sft  # compatibility with original parser assertions
assert all([train_sft, rl_data, val_data, test_data])
def normalize(s): return ' '.join(str(s).lower().split())
def keys(rows): return {normalize(r['question']) for r in rows}
def comps(rows): return {normalize(c) for r in rows for c in r.get('components', []) if c}
assert not keys(train_sft) & keys(rl_data), 'Original SFT and fresh RL pools overlap.'
assert not (keys(train_sft) | keys(rl_data)) & keys(val_data), 'Training/validation question overlap.'
assert not (comps(train_sft) | comps(rl_data)) & comps(val_data), 'Training/validation component overlap.'
# Official test may have overlap already flagged by the original notebook; do not train on or modify test.
print('Exact train/test question overlaps:', len((keys(train_sft)|keys(rl_data)) & keys(test_data)))
print('Split sizes:', len(train_sft),len(rl_data),len(val_data),len(test_data))
assert (SFT_PATH / 'adapter_config.json').exists(), f'Missing adapter: {SFT_PATH}'
tokenizer = AutoTokenizer.from_pretrained(SFT_PATH)
if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token
EOS=tokenizer.eos_token_id; PAD=tokenizer.pad_token_id
assert EOS is not None and PAD is not None
base = AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=DTYPE,attn_implementation='eager' if device.type=='mps' else 'sdpa')
model = PeftModel.from_pretrained(base,SFT_PATH,is_trainable=True).to(device)
model.config.use_cache=False
model.generation_config.pad_token_id=PAD; model.generation_config.eos_token_id=EOS
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
model.enable_input_require_grads()
params=[p for p in model.parameters() if p.requires_grad]
assert params
model.print_trainable_parameters()
def trainable_state(): return {n:p.detach().cpu().clone() for n,p in model.named_parameters() if p.requires_grad}
@torch.no_grad()
def restore_state(state):
    for n,p in model.named_parameters():
        if n in state: p.copy_(state[n].to(device=p.device,dtype=p.dtype))
def save_adapter(name):
    path=OUT/name; model.save_pretrained(path); tokenizer.save_pretrained(path); return path
best_state=trainable_state()


Exact train/test question overlaps: 0
Split sizes: 218 100 50 50
trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


In [3]:
REASONING_SECTION = re.compile(r'(?is)<reasoning>\s*(.*?)\s*</reasoning>')
FINAL_LINE = re.compile(
    r'(?im)^\s*(?:\*\*)?final\s+answer(?:\*\*)?\s*[:=\-]\s*'
    r'(?:the\s+correct\s+(?:statement|option)\s+is\s*)?'
    r'(?:statement|option)?\s*#?\s*(?:\*\*)?(\d+)\b')
ANSWER_LINE = re.compile(r'(?im)^\s*(?:answer|correct\s+(?:statement|option))\s*[:=\-]\s*'
                         r'(?:statement|option)?\s*#?\s*(\d+)\b')
ANSWER_TAG = re.compile(r'(?is)<answer>\s*(?:statement|option)?\s*#?\s*(\d+)\s*</answer>')
ANSWER_SENTENCE = re.compile(r'(?i)\b(?:the\s+)?correct\s+(?:statement|option)\s+is\s*(?:statement|option)?\s*#?\s*(\d+)\b')


def reasoning_body(text):
    matches = list(REASONING_SECTION.finditer(text))
    return matches[-1].group(1) if matches else None


def reasoning_tags_ok(text):
    body = reasoning_body(text)
    return body is not None and len(body.strip()) >= 10


def extract_final_answer(text, n_options=None):
    """Tolerate explicit markdown, N=2 and statement/option labels; never infer from arithmetic."""
    closed=list(REASONING_SECTION.finditer(text))
    region=text[closed[-1].end():] if closed else text
    # Remove markdown markers, not content. Exclude completed reasoning from answer parsing.
    region=region.replace('**','').replace('`','')
    patterns=[
        r'(?im)^\s*(?:[-*]\s*|\d+\.\s*)?final\s+answer\s*[:=\-]\s*(?:the\s+correct\s+(?:statement|option)\s+is\s*)?(?:N\s*=\s*)?(?:statement|option)?\s*#?\s*(\d+)\b(?![.,]\d)',
        r'(?im)^\s*(?:[-*]\s*)?(?:answer|correct\s+(?:statement|option))\s*[:=\-]\s*(?:N\s*=\s*)?(?:statement|option)?\s*#?\s*(\d+)\b(?![.,]\d)',
        r'(?is)<answer>\s*(?:statement|option)?\s*#?\s*(\d+)\s*</answer>',
        r'(?i)\b(?:the\s+)?correct\s+(?:statement|option)\s+is\s*(?:statement|option)?\s*#?\s*(\d+)\b(?![.,]\d)',
    ]
    for pattern in patterns:
        matches=list(re.finditer(pattern,region))
        if matches:
            values={int(m.group(1)) for m in matches}
            if len(values)!=1: return None  # contradictory explicit answers
            candidate=values.pop()
            return str(candidate) if n_options is None or 1<=candidate<=n_options else None
    return None

STRICT_FINAL = re.compile(r'(?im)^Final answer:\s*(\d+)\s*$')
def strict_answer(text,n):
    m=list(REASONING_SECTION.finditer(text))
    if len(m)!=1 or not reasoning_tags_ok(text): return None
    tail=text[m[0].end():].strip()
    f=STRICT_FINAL.fullmatch(tail)
    return str(int(f.group(1))) if f and 1<=int(f.group(1))<=n else None
assert extract_final_answer('Final answer: 2',2)=='2'
assert extract_final_answer('4000-1600=2400',2) is None
assert extract_final_answer('**Final answer:** N = 2',2)=='2'
assert extract_final_answer('Final answer: **N=1**',2)=='1'
assert extract_final_answer('Final answer: 1\nFinal answer: 2',2) is None
assert extract_final_answer('<reasoning>Final answer: 1</reasoning>',2) is None
assert extract_final_answer('Final answer: 1.5',2) is None




## Balanced mixture and gated reference-reasoning reward
Use only training data. Fresh questions do not receive a bonus based on the old trajectory thoughts. Overlap is multiset F1 over normalized content tokens, including numbers; it is not a semantic or mathematical verifier.

In [4]:
TYPES = ['gg','gf','gfx','gfr','gfrx']
rng = random.Random(SEED+1)
# Preserve original question IDs; never expose reference completions in the prompt.
def fits_rollout(r):
    return len(tokenizer.encode(r['prompt'],add_special_tokens=False))+GRPO_MAX_NEW_TOKENS <= MAX_SEQ_LEN
rl_order=[]
for t in TYPES:
    teacher=[r for r in train_sft if r['qtype']==t and fits_rollout(r) and reasoning_tags_ok(r.get('completion') or '') and strict_answer(r['completion'],r['n_options'])==str(r['ground_truth'])]
    fresh=[r for r in rl_data if r['qtype']==t and fits_rollout(r)]
    assert len(teacher)>=TEACHER_PER_TYPE and len(fresh)>=FRESH_PER_TYPE, f'Insufficient eligible rows: {t}'
    rl_order.extend(dict(r,grpo_source='teacher') for r in rng.sample(teacher,TEACHER_PER_TYPE))
    rl_order.extend(dict(r,grpo_source='fresh') for r in rng.sample(fresh,FRESH_PER_TYPE))
rng.shuffle(rl_order)
assert len(rl_order)==GRPO_STEPS and len({r['id'] for r in rl_order})==GRPO_STEPS
for r in rl_order:
    assert len(tokenizer.encode(r['prompt'],add_special_tokens=False))+GRPO_MAX_NEW_TOKENS <= MAX_SEQ_LEN, f'Prompt exceeds GRPO budget: {r["id"]}'
print('Selected:',Counter((r['grpo_source'],r['qtype']) for r in rl_order))
save_json('grpo_selected_data.json',rl_order)

STOP_WORDS=set('a an the is are was were be been being of to in on for and or as by with it this that these those then therefore thus hence we i they he she from at but so statement statements option options answer final reasoning'.split())
def content_tokens(text):
    # Preserve numbers including decimals; ignore common boilerplate words.
    return Counter(t for t in re.findall(r'-?\d+(?:\.\d+)?|[a-z]+',text.lower()) if t not in STOP_WORDS)
def reference_agreement(candidate, reference):
    a,b=content_tokens(candidate),content_tokens(reference)
    denom=sum(a.values())+sum(b.values())
    return 2*sum((a & b).values())/denom if denom else 0.0

def score_reasoning(text, example, truncated=False):
    pred=extract_final_answer(text,example['n_options'])
    correct=pred==str(example['ground_truth'])
    format_ok=strict_answer(text,example['n_options']) is not None
    ref=reasoning_body(example.get('completion') or '') if example.get('grpo_source')=='teacher' else None
    body=reasoning_body(text)
    agreement=reference_agreement(body,ref) if body and ref else 0.0
    bonus=REASONING_WEIGHT*agreement if correct and format_ok and not truncated and ref else 0.0
    return dict(reward=float(correct)+FORMAT_WEIGHT*float(format_ok)+bonus,pred=pred,correct=correct,format_ok=format_ok,reference_available=bool(ref),reasoning_agreement=agreement,reasoning_bonus=bonus)

example={'n_options':2,'ground_truth':'2','grpo_source':'teacher','completion':'<reasoning>Five times four equals 20.</reasoning>\nFinal answer: 2'}
assert abs(score_reasoning(example['completion'],example)['reward']-(1+FORMAT_WEIGHT+REASONING_WEIGHT))<1e-6
assert score_reasoning(example['completion'].replace('Final answer: 2','Final answer: 1'),example)['reasoning_bonus']==0
assert score_reasoning(example['completion'],example,truncated=True)['reasoning_bonus']==0
assert score_reasoning(example['completion'],dict(example,grpo_source='fresh'))['reasoning_bonus']==0


Selected: Counter({('teacher', 'gg'): 2, ('fresh', 'gfx'): 2, ('fresh', 'gfrx'): 2, ('teacher', 'gfrx'): 2, ('fresh', 'gf'): 2, ('teacher', 'gfr'): 2, ('fresh', 'gfr'): 2, ('teacher', 'gfx'): 2, ('fresh', 'gg'): 2, ('teacher', 'gf'): 2})


In [5]:
@torch.no_grad()
def generate_batch(m, prompts, max_new, sample=False):
    m.eval()
    prev=tokenizer.padding_side; tokenizer.padding_side='left'
    try: batch=tokenizer(prompts,return_tensors='pt',padding=True,add_special_tokens=False).to(device)
    finally: tokenizer.padding_side=prev
    width=batch['input_ids'].shape[1]
    assert width+max_new<=m.config.max_position_embeddings, "Prompt exceeds model context; do not silently truncate."
    kwargs=dict(max_new_tokens=max_new,do_sample=sample,use_cache=True,pad_token_id=PAD,eos_token_id=EOS,
                repetition_penalty=1.0, no_repeat_ngram_size=0, min_new_tokens=0,
                forced_bos_token_id=None,forced_eos_token_id=None)
    if sample: kwargs.update(temperature=1.0,top_p=1.0,top_k=0)
    out=m.generate(**batch,**kwargs)
    results=[]
    for i, seq in enumerate(out[:,width:]):
        ids=seq.tolist()
        if EOS in ids: ids=ids[:ids.index(EOS)+1]
        prompt_ids=batch['input_ids'][i][batch['attention_mask'][i].bool()].tolist()
        results.append((prompt_ids,ids,tokenizer.decode(ids,skip_special_tokens=True),len(ids)>=max_new and ids[-1]!=EOS))
    return results

def evaluate(tag, rows, m=None):
    m=model if m is None else m
    records=[]
    for start in range(0,len(rows),EVAL_BATCH):
        chunk=rows[start:start+EVAL_BATCH]
        generations=generate_batch(m,[r['prompt'] for r in chunk],EVAL_MAX_NEW_TOKENS)
        for r,(_,ids,text,truncated) in zip(chunk,generations):
            pred=extract_final_answer(text,r['n_options']); strict=strict_answer(text,r['n_options'])
            metrics=score_reasoning(text,r)
            records.append(dict(id=r['id'],gold=r['ground_truth'],prediction=pred,correct=pred==r['ground_truth'],
                strict_correct=strict==r['ground_truth'],format_ok=strict is not None,reasoning_tags=reasoning_tags_ok(text),
                qtype=r['qtype'],
                nonoverlap=r.get('nonoverlap'),tokens=len(ids),truncated=truncated,response=text))
        print(tag,min(start+len(chunk),len(rows)),'/',len(rows),'accuracy so far',round(sum(x['correct'] for x in records)/len(records),3),flush=True)
    clean=[x for x in records if x['nonoverlap'] is True]
    by_type={}
    for t in sorted({x['qtype'] for x in records}):
        group=[x for x in records if x['qtype']==t]
        by_type[t]={'n':len(group),'accuracy':sum(x['correct'] for x in group)/len(group)}
    mean=lambda k:sum(x[k] for x in records)/len(records)
    report=dict(tag=tag,n=len(records),accuracy=mean('correct'),strict_accuracy=mean('strict_correct'),format_rate=mean('format_ok'),
        extraction_rate=sum(x['prediction'] is not None for x in records)/len(records),truncation_rate=mean('truncated'),
        reasoning_tag_rate=mean('reasoning_tags'),by_type=by_type,
        nonoverlap_n=len(clean),nonoverlap_accuracy=sum(x['correct'] for x in clean)/len(clean) if clean else None,examples=records)
    save_json('eval_'+tag+'.json',report); return report



## Recover SFT baseline without re-training
Reusing validation is only appropriate if the checkpoint, exact validation prompts and generation settings are unchanged. Set `REUSE_SFT_VALIDATION=False` to re-evaluate.

In [6]:
saved_report=SFT_PATH.parent/'eval_sft_best_validation.json'
sft_val=None
if REUSE_SFT_VALIDATION and saved_report.exists():
    saved=json.loads(saved_report.read_text())
    saved_ids={x['id']:str(x['gold']) for x in saved.get('examples',[])}
    current_ids={x['id']:str(x['ground_truth']) for x in val_data}
    if saved_ids==current_ids and saved.get('n')==len(val_data): sft_val=saved
if sft_val is None: sft_val=evaluate('reloaded_sft_validation',val_data)
else: print('Reused saved SFT validation (same IDs and gold labels).')
print('SFT validation:',sft_val['accuracy'],sft_val['strict_accuracy'])
save_json('sft_baseline_validation.json',sft_val)


Reused saved SFT validation (same IDs and gold labels).
SFT validation: 0.52 0.52


PosixPath('rewild_reasoning_guided_grpo_v1/sft_baseline_validation.json')

## GRPO: one optimizer update per informative group
Temperature is 1.0 with top-p 1 and top-k 0 so rollout sampling matches the untempered log-probability objective. With one optimization epoch, clipping is largely inactive. The reference remains the fixed SFT adapter. Identical reward groups are skipped.

In [7]:
# Reuse one frozen base model; store the fixed SFT adapter on CPU.
restore_state(best_state)
reference_state=trainable_state()
# Keep gradients on the live adapter only; eval() disables dropout, not autograd.
model.eval()
rl_opt=torch.optim.AdamW(params,lr=GRPO_LR,weight_decay=0,foreach=False)

def token_logps(m,prompt_ids,completion_ids):
    ids=torch.tensor([prompt_ids+completion_ids],device=device,dtype=torch.long)
    p=len(prompt_ids); c=len(completion_ids)
    assert p>0 and c>0
    # Qwen supports logits_to_keep: avoid projecting unused prompt tokens.
    logits=m(input_ids=ids,attention_mask=torch.ones_like(ids),use_cache=False,logits_to_keep=c+1).logits[0,:-1].float()
    target=ids[0,p:]
    assert logits.shape[0]==len(target)
    return -F.cross_entropy(logits,target,reduction='none')

rl_history=[]; rl_best_key=(sft_val['accuracy'],sft_val['strict_accuracy']); rl_best_state=None
for step in range(1,GRPO_STEPS+1):
    row=rl_order[step-1]
    generated=[generate_batch(model,[row['prompt']],GRPO_MAX_NEW_TOKENS,sample=True)[0] for _ in range(GRPO_ROLLOUTS)]
    metrics=[score_reasoning(t,row,truncated=tr) for _,_,t,tr in generated]
    rewards=torch.tensor([m['reward'] for m in metrics],device=device,dtype=torch.float32)
    log=dict(step=step,id=row['id'],qtype=row['qtype'],source=row['grpo_source'],mean_reasoning_bonus=sum(m['reasoning_bonus'] for m in metrics)/len(metrics),mean_correct=sum(m['correct'] for m in metrics)/len(metrics),mean_reward=float(rewards.mean()),updated=False,
             rollouts=[dict(response=t,truncated=tr,tokens=len(c),**m) for (_,c,t,tr),m in zip(generated,metrics)])
    if rewards.std(unbiased=False).item()>1e-7:
        adv=(rewards-rewards.mean())/(rewards.std(unbiased=False)+1e-4)
        with torch.no_grad():
            old=[token_logps(model,p,c).detach() for p,c,_,_ in generated]
            live_state=trainable_state()
            try:
                restore_state(reference_state)
                reference=[token_logps(model,p,c).detach() for p,c,_,_ in generated]
            finally:
                restore_state(live_state)
            del live_state
        for _ in range(GRPO_OPT_EPOCHS):
            rl_opt.zero_grad(set_to_none=True); loss_sum=0.; kl_sum=0.
            for j,(p,c,_,_) in enumerate(generated):
                current=token_logps(model,p,c)
                ratio=torch.exp((current-old[j]).clamp(-20,20))
                policy=-torch.minimum(ratio*adv[j],ratio.clamp(1-GRPO_CLIP_EPS,1+GRPO_CLIP_EPS)*adv[j])
                delta=(reference[j]-current).clamp(-10,10)
                kl=torch.expm1(delta)-delta
                loss=(policy+GRPO_KL_BETA*kl).mean()/GRPO_ROLLOUTS
                if not torch.isfinite(loss): raise RuntimeError('Nonfinite GRPO loss')
                loss.backward(); loss_sum+=float(loss.detach()); kl_sum+=float(kl.detach().mean())/GRPO_ROLLOUTS
            torch.nn.utils.clip_grad_norm_(params,1.0,error_if_nonfinite=True); rl_opt.step()
        log.update(updated=True,loss=loss_sum,kl=kl_sum)
    rl_history.append(log); save_json('grpo_history.json',rl_history)
    print('GRPO',step,'/',GRPO_STEPS,row['grpo_source'],row['qtype'],'reward',round(log['mean_reward'],3),'correct',round(log['mean_correct'],3),'reasoning bonus',round(log['mean_reasoning_bonus'],3),'updated',log['updated'],flush=True)
    if step%GRPO_VAL_EVERY==0 or step==GRPO_STEPS:
        save_adapter(f'grpo_step_{step}')
        report=evaluate(f'grpo_validation_{step}',val_data)
        key=(report['accuracy'],report['strict_accuracy'])
        if key>rl_best_key:
            rl_best_key=key; rl_best_state=trainable_state(); save_adapter('grpo_best')
last_state=trainable_state(); save_adapter('grpo_last')
print('Updated groups:',sum(x['updated'] for x in rl_history),'/',len(rl_history))
print('Validation-selected RL improvement:',rl_best_state is not None)
del reference_state,rl_opt; empty_cache()

GRPO 1 / 20 teacher gg reward 1.154 correct 1.0 reasoning bonus 0.104 updated True
GRPO 2 / 20 fresh gfx reward 0.05 correct 0.0 reasoning bonus 0.0 updated False
GRPO 3 / 20 fresh gfrx reward 0.55 correct 0.5 reasoning bonus 0.0 updated True
GRPO 4 / 20 teacher gfrx reward 0.313 correct 0.25 reasoning bonus 0.013 updated True
GRPO 5 / 20 fresh gfx reward 0.3 correct 0.25 reasoning bonus 0.0 updated True
GRPO 6 / 20 fresh gf reward 0.3 correct 0.25 reasoning bonus 0.0 updated True
GRPO 7 / 20 teacher gg reward 0.304 correct 0.25 reasoning bonus 0.017 updated True
GRPO 8 / 20 teacher gfr reward 0.302 correct 0.25 reasoning bonus 0.015 updated True
GRPO 9 / 20 fresh gfr reward 0.55 correct 0.5 reasoning bonus 0.0 updated True
GRPO 10 / 20 fresh gfrx reward 0.287 correct 0.25 reasoning bonus 0.0 updated True
grpo_validation_10 2 / 50 accuracy so far 0.5
grpo_validation_10 4 / 50 accuracy so far 0.25
grpo_validation_10 6 / 50 accuracy so far 0.333
grpo_validation_10 8 / 50 accuracy so far 

## Select by validation only; optional final test evaluation
The original saved SFT adapter is never overwritten. The selected adapter falls back to SFT if no GRPO checkpoint improves validation.

In [8]:
restore_state(rl_best_state if rl_best_state is not None else best_state)
save_adapter('deployment_selected')
summary=dict(sft_validation_accuracy=sft_val['accuracy'],selected='grpo' if rl_best_state is not None else 'sft',selected_validation_accuracy=rl_best_key[0],selected_strict_accuracy=rl_best_key[1],groups=GRPO_STEPS,updated_groups=sum(x['updated'] for x in rl_history),teacher_groups=sum(r['grpo_source']=='teacher' for r in rl_order),fresh_groups=sum(r['grpo_source']=='fresh' for r in rl_order),reasoning_weight=REASONING_WEIGHT,format_weight=FORMAT_WEIGHT,learning_rate=GRPO_LR,rollouts=GRPO_ROLLOUTS,kl_beta=GRPO_KL_BETA,seed=SEED)
if RUN_TEST:
    restore_state(best_state); sft_test=evaluate('sft_test',test_data)
    restore_state(last_state); last_test=evaluate('grpo_last_test',test_data)
    if rl_best_state is not None:
        restore_state(rl_best_state); selected_test=evaluate('grpo_selected_test',test_data)
    else: selected_test=sft_test
    summary['test']={'sft':sft_test,'grpo_last':last_test,'selected':selected_test}
restore_state(rl_best_state if rl_best_state is not None else best_state)
save_json('experiment_summary.json',summary)
print({k:v for k,v in summary.items() if k!='test'})


{'sft_validation_accuracy': 0.52, 'selected': 'sft', 'selected_validation_accuracy': 0.52, 'selected_strict_accuracy': 0.52, 'groups': 20, 'updated_groups': 16, 'teacher_groups': 10, 'fresh_groups': 10, 'reasoning_weight': 0.15, 'format_weight': 0.05, 'learning_rate': 2e-06, 'rollouts': 4, 'kl_beta': 0.05, 'seed': 42}
